# Assignment · Representaciones de lenguaje: BoW y Word2Vec

**Sesión 1 · Asignment**
**Clase: Modelos de Lenguaje de Gran Escala**
**Universidad de La Sabana**
**Autor: Andres Rosso**  
**Estudiante: Alejandro Moncada**

**Objetivo:** construir desde cero las principales herramientas usadas en clase sobre `BBC news.csv`.

En este assignment se entrega **la estructura y los nombres de las funciones**, pero ustedes deben implementar la mayor parte de la solución.

Pueden reutilizar ideas del notebook de clase, pero deben entender y adaptar el código.

> Entreguen el notebook ejecutado, con resultados, gráficas y respuestas breves.


## 0. Preparación

Carguen las librerías necesarias y `BBC news.csv`.

Al terminar deben tener un `DataFrame` llamado `df`.


Este notebook es una plantilla: las celdas marcadas con `TODO` deben completarse antes de ejecutar las comprobaciones que dependen de ellas. Para cargar el archivo, pueden tomar como referencia la preparación del notebook de clase.

In [ ]:
import random
import re
import zlib
from collections import Counter
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RUTA_CSV = "BBC news.csv"

df = pd.read_csv(RUTA_CSV)

display(df.head())
print(df.shape)

### EDA del corpus

Antes de tokenizar reviso qué trae el archivo y si hay algo que afecte las decisiones de la sección 1.

In [ ]:
texts = df["content"]
years = df["publication_date"].str.strip('"').str[:4]
chars = texts.str.len()

display(pd.DataFrame({
    "Comprobación": [
        "Nulos en content",
        "Noticias con content duplicado",
        "URLs duplicadas",
        "Nulos en keyword",
        "Noticias con menos de 500 caracteres",
        "Separadores ';' en el texto",
        "Separadores ';' justo después de punto",
        "Apóstrofes tipográficos (’)",
    ],
    "Valor": [
        texts.isna().sum(),
        texts.duplicated().sum(),
        df["url"].duplicated().sum(),
        df["keyword"].isna().sum(),
        (chars < 500).sum(),
        texts.str.count(";").sum(),
        texts.str.count(r"\.;").sum(),
        texts.str.count("’").sum(),
    ],
}))
display(chars.describe().round(0).to_frame("Caracteres por noticia"))
display(years.value_counts().sort_index().to_frame("Noticias").T)

top_10_docs_share = chars.sort_values().tail(10).sum() / chars.sum()
print(f"Las 10 noticias más largas concentran {top_10_docs_share:.1%} de los caracteres.")
print("Secciones más comunes en la URL:")
sections = df["url"].str.extract(r"bbc\.com/([a-z]+/[a-z]+)")[0]
print(sections.value_counts().head(5).to_dict())

**Lo que encontré:**

- Trabajo con `content`, la única columna con el cuerpo de la noticia. No tiene nulos ni duplicados, así que se quedan las 1.000 noticias.
- Las longitudes son muy dispares: la mediana es 2.114 caracteres y la más larga llega a 57.732. Solo 10 noticias concentran el 7,7 % del texto.
- El corpus es reciente y desbalanceado en el tiempo: 918 noticias son de 2021 a 2024.
- No hay columna de categoría. `keyword` está vacía en 977 filas, así que no comparo por categorías.
- El archivo une párrafos con `; `: de 15.017 punto y comas, 12.679 vienen justo después de un punto. Por eso los trato como fin de oración.

## 1. Tokenización y preparación del corpus · 20%

Implementen las mismas decisiones de tokenización utilizadas en clase:

- convertir a minúsculas;
- conservar contracciones en inglés;
- permitir decidir si se conservan números;
- separar el texto en oraciones;
- construir las columnas `oraciones`, `tokens` y `longitud`.

No usen un tokenizador externo.


In [ ]:
STOP = set(ENGLISH_STOP_WORDS)

SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+|;\s+")


def tokenizar(texto, minusculas=True, numeros=False):
    if minusculas:
        texto = texto.lower()
    chars = "a-z" if minusculas else "a-zA-Z"
    if numeros:
        chars += "0-9"
    return re.findall(rf"[{chars}]+(?:'[{chars}]+)*", texto)


def oraciones_tokenizadas(texto):
    oraciones = (tokenizar(o) for o in SENTENCE_SPLIT.split(texto))
    return [o for o in oraciones if o]


df["oraciones"] = df["content"].map(oraciones_tokenizadas)
df["tokens"] = df["content"].map(tokenizar)
df["longitud"] = df["tokens"].map(len)

FRECUENCIAS = Counter(t for tokens in df["tokens"] for t in tokens)

assert df["oraciones"].map(lambda o: sum(map(len, o))).equals(df["longitud"])

### Comprobación

Su implementación debe permitir responder con código:

1. ¿Cuántas noticias hay?
2. ¿Cuántos tokens tiene el corpus?
3. ¿Cuántas palabras distintas aparecen?
4. ¿Cuáles son las 10 palabras más frecuentes sin *stop words*?


In [ ]:
n_noticias = len(df)
n_tokens = int(df["longitud"].sum())
n_palabras = len(FRECUENCIAS)
top_10 = [(w, f) for w, f in FRECUENCIAS.most_common() if w not in STOP][:10]

print("1. Noticias:", n_noticias)
print("2. Tokens:", n_tokens)
print("3. Palabras distintas:", n_palabras)
print("4. Top 10 sin stop words:", top_10)

In [ ]:
REFERENCIA = {"Noticias": 1000, "Tokens": 523000, "Palabras distintas": 25600}
obtenido = {"Noticias": n_noticias, "Tokens": n_tokens, "Palabras distintas": n_palabras}

verificacion = pd.DataFrame({"Referencia de clase": REFERENCIA, "Obtenido": obtenido})
verificacion["Diferencia %"] = (
    (verificacion["Obtenido"] / verificacion["Referencia de clase"] - 1) * 100
).round(2)
display(verificacion)
assert (verificacion["Diferencia %"].abs() <= 1).all()


def measure(name, tokenize):
    counts = Counter(t for text in df["content"] for t in tokenize(text))
    return {"Variante": name, "Tokens": sum(counts.values()), "Palabras distintas": len(counts)}


variantes = pd.DataFrame([
    measure("Elegida: minúsculas, sin números, contracciones", tokenizar),
    measure("Conservando números", lambda t: tokenizar(t, numeros=True)),
    measure("Sin pasar a minúsculas", lambda t: tokenizar(t, minusculas=False)),
    measure("Normalizando ’ a '", lambda t: tokenizar(t.replace("’", "'"))),
    measure("Sin contracciones", lambda t: re.findall(r"[a-z]+", t.lower())),
])
variantes["Δ tokens"] = variantes["Tokens"] - n_tokens
variantes["Δ palabras"] = variantes["Palabras distintas"] - n_palabras
display(variantes)

stop_tokens = sum(f for w, f in FRECUENCIAS.items() if w in STOP)
print(f"Las stop words son {stop_tokens / n_tokens:.1%} de los tokens y se conservan en el corpus.")
print("Ejemplo del apóstrofe tipográfico:", tokenizar("don’t and don't"))
print("Frecuencia de los fragmentos 's' y 't':", FRECUENCIAS["s"], FRECUENCIAS["t"])

display(df["longitud"].describe().round(0).to_frame("Tokens por noticia"))
print("Oraciones en el corpus:", int(df["oraciones"].map(len).sum()))

plt.figure(figsize=(7, 3))
plt.hist(df["longitud"], bins=np.logspace(np.log10(df["longitud"].min()), np.log10(df["longitud"].max()), 30))
plt.xscale("log")
plt.xlabel("Tokens por noticia (escala log)")
plt.ylabel("Noticias")
plt.title("Longitud de las noticias")
plt.show()

**Decisiones de preprocesamiento.** Los conteos coinciden con la referencia de clase (−0,05 % en tokens y +0,06 % en vocabulario), y la tabla de variantes muestra cuánto pesa cada decisión:

- **Minúsculas:** no cambian los tokens, pero sin ellas el vocabulario sube en 3.285 palabras porque `The` y `the` contarían aparte.
- **Números:** fuera por defecto. Incluirlos suma 8.498 tokens y 1.257 palabras, y el análisis no se apoya en cifras. Se pueden activar con `numeros=True`.
- **Contracciones:** se conservan como un solo token (`don't`), como se pidió. Separarlas sumaría 5.055 tokens.
- **Apóstrofe tipográfico:** no lo normalizo, porque así se reproducen los conteos de clase. Tiene un costo visible: `don’t` se parte en `don` y `t`, y el fragmento `s` aparece 1.370 veces, tanto que entra al top 10. Normalizarlo restaría 1.731 tokens y sumaría 139 palabras, también dentro del 1 %.
- **Stop words:** son el 47,7 % de los tokens, pero se quedan en el corpus porque las concordancias y Word2Vec necesitan la oración completa. Solo se ocultan en tablas de frecuencias, bigramas y BoW.
- **Límites del tokenizador:** solo reconoce letras de la `a` a la `z`, así que una palabra con tilde se corta, y una abreviatura con punto parte la oración.

Todas las secciones usan estos mismos documentos y estas mismas columnas.

## 2. Frecuencias, bigramas y Zipf · 20%

Implemente una función que permita explorar las palabras o bigramas más frecuentes.

Debe:

- recibir un número `n`;
- permitir incluir o excluir *stop words*;
- permitir elegir entre palabras y bigramas;
- devolver un `DataFrame` ordenado por frecuencia.

No es necesario crear widgets.


In [ ]:
BIGRAMAS = Counter(
    bigrama
    for oraciones in df["oraciones"]
    for oracion in oraciones
    for bigrama in zip(oracion, oracion[1:])
)


def explorar_frecuencias(n=10, ocultar_stopwords=True, unidad="Palabras"):
    if unidad == "Palabras":
        items = FRECUENCIAS.items()
        keep = lambda w: w not in STOP
        show = str
    elif unidad == "Bigramas":
        items = BIGRAMAS.items()
        keep = lambda b: b[0] not in STOP and b[1] not in STOP
        show = " ".join
    else:
        raise ValueError("unidad debe ser 'Palabras' o 'Bigramas'")
    filas = [(show(k), f) for k, f in items if not ocultar_stopwords or keep(k)]
    filas.sort(key=lambda fila: (-fila[1], fila[0]))
    return pd.DataFrame(filas[:n], columns=[unidad[:-1], "Frecuencia"])


# Pruebas mínimas
display(explorar_frecuencias(10, True, "Palabras"))
display(explorar_frecuencias(10, True, "Bigramas"))

In [ ]:
display(explorar_frecuencias(10, False, "Palabras"))
display(explorar_frecuencias(10, False, "Bigramas"))

Con stop words, la tabla es casi pura gramática (`of the`, `in the`). Sin ellas se ve de qué habla el corpus: `mr musk` (508), `junior doctors` (216), `dangerous driving` (155).

También se ve cómo está hecho el archivo: `follow bbc` (361), `story ideas` (355) y `facebook twitter` (169) vienen del texto repetido al final de las notas, no de la noticia. No lo quito porque cambiaría los conteos de referencia, pero conviene tenerlo presente al leer las frecuencias.

### Zipf

Implemente una función que grafique **frecuencia vs. rango**.

El gráfico debe:

- ordenar las frecuencias de mayor a menor;
- usar escala logarítmica en ambos ejes;
- mostrar rango en X y frecuencia en Y.

Después responda: **¿el corpus presenta aproximadamente el patrón esperado por Zipf?**


In [ ]:
def fit_zipf(freqs, start, end):
    ranks = np.arange(start, end + 1)
    x, y = np.log10(ranks), np.log10(freqs[start - 1:end])
    slope, intercept = np.polyfit(x, y, 1)
    r2 = 1 - ((y - (slope * x + intercept)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return slope, intercept, r2


def graficar_zipf():
    freqs = np.array(sorted(FRECUENCIAS.values(), reverse=True))
    ranks = np.arange(1, len(freqs) + 1)
    slope, intercept, r2 = fit_zipf(freqs, 1, len(freqs))
    plt.figure(figsize=(7, 4.5))
    plt.loglog(ranks, freqs, ".", markersize=3, label="Corpus")
    plt.loglog(ranks, freqs[0] / ranks, "--", label="Zipf ideal (pendiente -1)")
    plt.loglog(ranks, 10 ** intercept * ranks ** slope, ":", label=f"Ajuste (pendiente {slope:.2f}, R² {r2:.3f})")
    plt.xlabel("Rango")
    plt.ylabel("Frecuencia")
    plt.title("Ley de Zipf en el corpus")
    plt.legend()
    plt.show()


graficar_zipf()

In [ ]:
freqs = np.array(sorted(FRECUENCIAS.values(), reverse=True))
tramos = [(1, 10), (11, 1000), (1001, 10000), (10001, len(freqs)), (1, len(freqs))]
filas = []
for start, end in tramos:
    slope, _, r2 = fit_zipf(freqs, start, end)
    filas.append({
        "Rango": f"{start}-{end}",
        "Pendiente": round(slope, 2),
        "R²": round(r2, 3),
        "% de tokens": round(freqs[start - 1:end].sum() / n_tokens * 100, 1),
    })
display(pd.DataFrame(filas))

hapax = (freqs == 1).sum()
print(f"Palabras con frecuencia 1: {hapax} ({hapax / len(freqs):.1%} del vocabulario, {hapax / n_tokens:.2%} de los tokens)")
print("Frecuencia × rango en los primeros rangos:", [(r, int(freqs[r - 1] * r)) for r in (1, 2, 5, 10, 100, 1000, 10000)])

**Respuesta:**  
Sí, aproximadamente, y sobre todo en el centro de la distribución. Entre los rangos 11 y 1.000 la pendiente es −0,95 con R² de 0,999, muy cerca del −1 esperado, y ese tramo reúne el 50,8 % de los tokens.

En los extremos se desvía. Las 10 palabras más frecuentes caen más suave (−0,77) y ya suman el 21,9 % de los tokens. Desde el rango 1.001 la caída es más rápida (−1,39 y −1,29). Por eso el ajuste global da −1,35 aunque su R² sea 0,974: una sola recta esconde esa curvatura. Frecuencia × rango tampoco es constante: en los rangos revisados se mueve entre 28.565 y 66.000.

La cola se ve en escalones porque muchas palabras empatan: 10.428 aparecen una sola vez, el 40,7 % del vocabulario pero apenas el 1,99 % de los tokens.

## 3. Bag of Words · 20%

Implemente una función que construya una representación **Bag of Words** para los primeros `n_documentos`.

La función debe recibir:

- número de documentos;
- tamaño máximo del vocabulario;
- si se eliminan o no *stop words*.

Debe devolver un `DataFrame` documento-término.


In [ ]:
def bow_from_tokens(documentos, max_palabras=None, eliminar_stopwords=False):
    conteos = [Counter(t for t in doc if not eliminar_stopwords or t not in STOP) for doc in documentos]
    total = sum(conteos, Counter())
    vocabulario = [w for w, _ in sorted(total.items(), key=lambda kv: (-kv[1], kv[0]))][:max_palabras]
    return pd.DataFrame(
        [[c[w] for w in vocabulario] for c in conteos],
        columns=vocabulario,
        index=[f"Doc {i}" for i in range(len(documentos))],
    )


def construir_bow(n_documentos=5, max_palabras=15, eliminar_stopwords=True):
    documentos = df["tokens"].iloc[:n_documentos].tolist()
    return bow_from_tokens(documentos, max_palabras, eliminar_stopwords)


bow = construir_bow()
display(bow)

### Prueba de la representación

Utilicen su implementación para demostrar con código qué ocurre con:

`The student corrects the teacher`

`The teacher corrects the student`

Muestren ambos vectores y comprueben si son iguales.


In [ ]:
frases = [
    "The student corrects the teacher",
    "The teacher corrects the student"
]

tokens_frases = [tokenizar(f) for f in frases]
bow_frases = bow_from_tokens(tokens_frases)
bow_frases.index = frases
display(bow_frases)

iguales = bow_frases.iloc[0].equals(bow_frases.iloc[1])
print("¿Los dos vectores BoW son iguales?", iguales)
assert iguales

bigramas_frases = [set(zip(t, t[1:])) for t in tokens_frases]
print("¿Los bigramas son iguales?", bigramas_frases[0] == bigramas_frases[1])
print("Bigramas que solo están en la primera frase:", sorted(bigramas_frases[0] - bigramas_frases[1]))

noticia = df["tokens"].iloc[0]
barajada = random.Random(SEED).sample(noticia, len(noticia))
print("Noticia 0 original:", " ".join(noticia[:10]), "...")
print("Noticia 0 barajada:", " ".join(barajada[:10]), "...")
print("¿Mismo vector BoW con las palabras barajadas?", Counter(noticia) == Counter(barajada))

**Explique en máximo 3 líneas qué información conserva BoW y qué información pierde.**

BoW conserva qué palabras aparecen en cada documento y cuántas veces.  
Pierde el orden y, con él, quién hace qué: las dos frases dan el mismo vector aunque digan lo contrario, y la noticia 0 barajada también.  
Los bigramas recuperan algo de eso: `student corrects` solo está en la primera frase.

## 4. Concordancias y contexto · 15%

Construyan primero un índice que permita localizar las apariciones de cada palabra.

Después implementen `concordancias()`.

La función debe recibir una palabra, tamaño de ventana y cantidad máxima de resultados, y devolver:

`Documento | Contexto izquierdo | Palabra | Contexto derecho`


In [ ]:
INDICE = {}
for doc, tokens in zip(df.index, df["tokens"]):
    for pos, token in enumerate(tokens):
        INDICE.setdefault(token, []).append((doc, pos))


def concordancias(palabra, ventana=5, cantidad=8, por_documento=None):
    palabra = palabra.lower()
    filas = []
    usados = Counter()
    for doc, pos in INDICE.get(palabra, []):
        if len(filas) == cantidad:
            break
        if por_documento is not None and usados[doc] == por_documento:
            continue
        usados[doc] += 1
        tokens = df.at[doc, "tokens"]
        filas.append({
            "Documento": doc,
            "Contexto izquierdo": " ".join(tokens[max(0, pos - ventana):pos]),
            "Palabra": tokens[pos],
            "Contexto derecho": " ".join(tokens[pos + 1:pos + 1 + ventana]),
        })
    return pd.DataFrame(filas, columns=["Documento", "Contexto izquierdo", "Palabra", "Contexto derecho"])


display(concordancias("government"))

Elijan **dos palabras diferentes**, muestren sus concordancias y expliquen brevemente qué información sobre su uso pueden inferir de los contextos.


In [ ]:
pd.set_option("display.max_colwidth", None)
PALABRAS_CONTEXTO = ["strike", "market"]
for palabra in PALABRAS_CONTEXTO:
    print(f"{palabra}: {FRECUENCIAS[palabra]} apariciones en {len({d for d, _ in INDICE[palabra]})} noticias")
    display(concordancias(palabra, ventana=5, cantidad=8, por_documento=1))

In [ ]:
def coocurrencias(palabra, ventana):
    conteo = Counter()
    for doc, pos in INDICE.get(palabra, []):
        tokens = df.at[doc, "tokens"]
        contexto = tokens[max(0, pos - ventana):pos] + tokens[pos + 1:pos + 1 + ventana]
        conteo.update(t for t in contexto if t not in STOP and t != palabra)
    return conteo


VENTANAS = [2, 5, 10]
for palabra in PALABRAS_CONTEXTO:
    conteos = {v: coocurrencias(palabra, v) for v in VENTANAS}
    display(pd.DataFrame({
        f"±{v}": [f"{w} ({f})" for w, f in conteos[v].most_common(10)] for v in VENTANAS
    }).rename_axis(palabra))
    tops = {v: {w for w, _ in conteos[v].most_common(50)} for v in VENTANAS}
    print(
        f"{palabra}: palabras de contexto distintas por ventana {[len(conteos[v]) for v in VENTANAS]}; "
        f"del top 50 compartidas ±2/±5: {len(tops[2] & tops[5])}, ±2/±10: {len(tops[2] & tops[10])}"
    )

**Respuesta:**  
**`strike`** (208 apariciones en 55 noticias) es en las filas vistas una huelga: `junior doctors strike`, `strike action`, `balloted on strike`. Funciona como sustantivo y como verbo (`whether to strike`), y sus contextos se parecen entre sí porque 44 de esas 55 noticias también mencionan `doctors`.

**`market`** (236 apariciones en 110 noticias) es más dispersa: `stock market`, `mass market`, `rental market`, `market value`. Lo que aclara el sentido es la palabra de al lado, porque puede hablar de bolsa, de carros o de vivienda.

**Efecto de la ventana.** Con ±2 aparece la colocación: `action` junto a `strike` (64), y `stock` (16) y `value` (13) junto a `market`. Con ±10 aparece el tema: alrededor de `strike`, `doctors` pasa de 11 a 88 y entran `pay` (36) y `bma` (35). También entra ruido: en `market` suben `said`, `mr` y `says`, que no dicen nada del mercado.

Las palabras de contexto distintas crecen mucho con la ventana (`strike`: 159, 416 y 730; `market`: 319, 823 y 1.456), y de las 50 más frecuentes con ±2 solo 19 y 26 siguen entre las 50 de ±10. Esto conecta con la hipótesis distribucional: conocemos una palabra por su compañía, y la ventana decide qué compañía miramos. Cerca es más bien sintaxis; lejos, tema.

## 5. Word2Vec · 20%

Implemente una función para entrenar Word2Vec sobre las oraciones del corpus.

Parámetros mínimos:

- `vector_size=50`
- `min_count=5`
- `workers=1`
- `seed=42`

El tamaño de la ventana debe ser un parámetro de la función.


Además de los parámetros pedidos, uso los valores por defecto de gensim (CBOW y 5 épocas) y agrego `modelo_10`. Paso un `hashfxn` estable para que los vectores no dependan de `PYTHONHASHSEED`.

Durante el entrenamiento, gensim 4.4 imprime en rojo `Exception ignored in: ...our_dot_float`. No es un error del notebook: lo comprobé entrenando dos veces y con tres combinaciones de NumPy y SciPy, y los vectores salieron idénticos.

In [ ]:
ORACIONES = [oracion for oraciones in df["oraciones"] for oracion in oraciones]


def stable_hash(texto):
    return zlib.crc32(texto.encode("utf-8"))


def entrenar_word2vec(ventana, seed=42):
    return Word2Vec(
        sentences=ORACIONES,
        vector_size=50,
        window=ventana,
        min_count=5,
        workers=1,
        seed=seed,
        hashfxn=stable_hash,
    )


modelo_2 = entrenar_word2vec(2)
modelo_5 = entrenar_word2vec(5)
modelo_10 = entrenar_word2vec(10)
tokens_modelo = sum(FRECUENCIAS[w] for w in modelo_2.wv.index_to_key)
print("Vocabulario de los modelos:", len(modelo_2.wv), "de", n_palabras, "palabras")
print(f"Esas palabras cubren {tokens_modelo / n_tokens:.1%} de los tokens.")
print(f"Tokens por oración: media {np.mean(list(map(len, ORACIONES))):.1f}, mediana {np.median(list(map(len, ORACIONES))):.0f}")

### Vecinos

Implemente una función que reciba una palabra, un modelo y `n`, y devuelva sus vecinos más cercanos con su similitud.


In [ ]:
def vecinos(palabra, modelo, n=5):
    palabra = palabra.lower()
    if palabra not in modelo.wv:
        return pd.DataFrame(columns=["Vecino", "Similitud"])
    return pd.DataFrame(modelo.wv.most_similar(palabra, topn=n), columns=["Vecino", "Similitud"]).round(3)


PALABRA = "government"

display(vecinos(PALABRA, modelo_2))
display(vecinos(PALABRA, modelo_5))
display(vecinos(PALABRA, modelo_10))

In [ ]:
MODELOS = {2: modelo_2, 5: modelo_5, 10: modelo_10}
PALABRAS_COMPARAR = ["government", "strike", "market", "doctors", "tesla", "good"]

for palabra in PALABRAS_COMPARAR:
    display(pd.DataFrame({
        f"Ventana {v}": [f"{w} ({s:.2f})" for w, s in m.wv.most_similar(palabra, topn=8)]
        for v, m in MODELOS.items()
    }).rename_axis(f"{palabra} ({FRECUENCIAS[palabra]})"))

In [ ]:
SEMILLAS = [42, 7, 2026]
modelos_semilla = {(v, s): entrenar_word2vec(v, seed=s) for v in VENTANAS for s in SEMILLAS}


def neighbors(palabra, modelo, n=10):
    return {w for w, _ in modelo.wv.most_similar(palabra, topn=n)}


def jaccard(a, b):
    return len(a & b) / len(a | b)


def mean_overlap(palabras, pares):
    return float(np.mean([
        jaccard(neighbors(p, modelos_semilla[a]), neighbors(p, modelos_semilla[b]))
        for p in palabras for a, b in pares
    ]))


# Same seed, different window: what the window changes (plus noise).
pares_ventana = [((a, s), (b, s)) for a, b in combinations(VENTANAS, 2) for s in SEMILLAS]
# Same window, different seed: pure training noise.
pares_semilla = [((v, a), (v, b)) for v in VENTANAS for a, b in combinations(SEMILLAS, 2)]

candidatas = sorted(
    w for w in modelo_2.wv.index_to_key
    if w not in STOP and len(w) >= 3 and "'" not in w
)
franjas = {"Alta (200+)": (200, 10**9), "Media (50-199)": (50, 199), "Baja (5-19)": (5, 19)}
rng = random.Random(SEED)
filas = []
for nombre, (low, high) in franjas.items():
    grupo = [w for w in candidatas if low <= FRECUENCIAS[w] <= high]
    muestra = rng.sample(grupo, min(30, len(grupo)))
    filas.append({
        "Frecuencia": nombre,
        "Palabras en la franja": len(grupo),
        "Palabras medidas": len(muestra),
        "Jaccard entre ventanas": round(mean_overlap(muestra, pares_ventana), 3),
        "Jaccard entre semillas": round(mean_overlap(muestra, pares_semilla), 3),
    })
display(pd.DataFrame(filas))

por_par = [{
    "Ventanas": f"{a} vs {b}",
    "Jaccard": round(mean_overlap(
        [w for w in candidatas if FRECUENCIAS[w] >= 200],
        [((a, s), (b, s)) for s in SEMILLAS],
    ), 3),
} for a, b in combinations(VENTANAS, 2)]
display(pd.DataFrame(por_par))

Compare los resultados de las ventanas 2 y 5.

**¿Qué cambió? ¿Los vecinos encontrados tienen que ser sinónimos? Explique usando al menos un resultado de sus modelos.**

**Qué cambió.** `government` conserva a `bma` y `trust`, pero con ventana 2 trae `minister` y `nuffield`, y con ventana 5 trae `union`, `vaccine` y `welsh`. `doctors` lo muestra más claro: con ventana 2 sus vecinos son `doctor`, `consultants`, `nhs` y `staff`, palabras que podrían ocupar un lugar parecido en la oración. Con ventana 10 son `action`, `strike` y `junior`: ya no es quién es, sino de qué se habla.

**Eso no es azar, en las palabras frecuentes.** Mido cuántos de los 10 vecinos se repiten (Jaccard). En palabras con 200 o más apariciones da 0,314 entre ventanas y 0,504 si solo cambio la semilla, y el cambio crece con la distancia entre ventanas (0,298 de 2 a 5, 0,174 de 2 a 10).

**La frecuencia manda sobre esa lectura.** En palabras con 5 a 19 apariciones, los vecinos casi no se repiten ni cambiando solo la semilla (0,048). Ahí no se puede atribuirle nada a la ventana. `min_count=5` deja 7.924 de las 25.616 palabras, que cubren el 94,3 % de los tokens.

**No tienen que ser sinónimos.** `bad` es el primer vecino de `good` con ventana 2 (0,94), y `sell` es el primero de `buy` con ventana 5 (0,94). Son opuestos, pero aparecen en los mismos contextos.

## 6. Integración · 5%

Elija **una palabra** y analícela usando todo lo construido:

1. frecuencia;
2. concordancias;
3. vecinos Word2Vec con ventana 2;
4. vecinos Word2Vec con ventana 5.

Muestre los resultados con código.


In [ ]:
PALABRA = "doctors"

print(f"Frecuencia de '{PALABRA}':", FRECUENCIAS[PALABRA])
print("Noticias donde aparece:", len({d for d, _ in INDICE[PALABRA]}))
print("Rango en el corpus:", sorted(FRECUENCIAS, key=FRECUENCIAS.get, reverse=True).index(PALABRA) + 1)

display(concordancias(PALABRA))

display(vecinos(PALABRA, modelo_2))

display(vecinos(PALABRA, modelo_5))

display(vecinos(PALABRA, modelo_10))

**Lectura de `doctors`.** Con 444 apariciones está en la franja donde los vecinos son más estables, pero se concentra en 75 noticias, y `junior doctors` aparece 216 veces. Las concordancias ya anticipan lo que hace el modelo: con ventana 2 sale el oficio (`doctor`, `consultants`, `staff`) y con ventana 10 sale la huelga (`action`, `strike`). Eso no significa que `doctors` y `strike` signifiquen lo mismo, sino que en este corpus suelen salir juntas.

In [ ]:
PARES = [
    ("good", "bad"), ("increase", "cut"), ("rise", "fall"), ("men", "women"),
    ("buy", "sell"), ("high", "low"), ("before", "after"), ("doctors", "nurses"),
]
filas = []
for a, b in PARES:
    fila = {"Par": f"{a} / {b}"}
    for v, m in MODELOS.items():
        fila[f"Ventana {v}"] = round(float(m.wv.similarity(a, b)), 3)
    fila["Rango de b entre los vecinos de a (v5)"] = next(
        (i for i, (w, _) in enumerate(modelo_5.wv.most_similar(a, topn=len(modelo_5.wv)), 1) if w == b), None
    )
    filas.append(fila)
display(pd.DataFrame(filas))
print("Noticias que mencionan a Musk:", sum("musk" in t for t in df["tokens"]))
print("Similitud tesla / billionaire (v5):", round(float(modelo_5.wv.similarity("tesla", "billionaire")), 3))
print("Similitud government / bma (v5):", round(float(modelo_5.wv.similarity("government", "bma")), 3))
for palabra in ["musk", "twitter", "immigration", "women"]:
    print(palabra, FRECUENCIAS[palabra], [w for w, _ in modelo_5.wv.most_similar(palabra, topn=8)])

### Análisis crítico

La cercanía entre vectores solo dice que dos palabras se usaron en contextos parecidos dentro de estas 1.000 noticias.

- **No es sinonimia.** `good`/`bad` y `buy`/`sell` quedan por encima de 0,92 en las tres ventanas. `government` y `bma` están a 0,85 con ventana 5, y son las dos partes de una disputa salarial.
- **No es verdad.** `tesla` queda a 0,82 de `billionaire`, y Tesla es una empresa. El modelo aprendió que la cobertura los pone juntos, no un hecho sobre Tesla. Los vecinos de `musk` incluyen a `trump` por lo mismo.
- **Refleja lo que se cubrió.** Musk aparece en 123 de las 1.000 noticias, y 44 de las 75 noticias con `doctors` también mencionan `strike`. Otro corpus daría otros vecinos, sin que el mundo hubiera cambiado.
- **La frecuencia limita lo que se puede decir.** `immigration` (21 apariciones) tiene vecinos como `sport`, `aj` y `f`. Incluso `women` (238) trae `rabbits`.
- **La similitud no es una probabilidad.** Los primeros vecinos de `market` llegan a 0,93 o más aunque mezclan `grid` o `longevity`. Un valor alto no garantiza una relación con sentido.
- **Cada palabra tiene un solo vector.** Todos los usos de `market` quedan promediados. Además, las oraciones tienen 22 tokens en promedio, así que la ventana 10 ya roza el límite de la oración.

### Conclusión

En máximo **5 líneas**, compare las dos representaciones principales trabajadas:

- **Bag of Words**
- **Word2Vec**

Explique qué representa cada una, qué información utiliza y una limitación de cada representación.

**Respuesta:**  
Bag of Words representa cada noticia como conteos de palabras: usa solo qué aparece y cuánto, y se lee fácil.  
Su limitación es que pierde el orden: `The student corrects the teacher` y su inversa dan el mismo vector.  
Word2Vec representa cada palabra con un vector denso aprendido de sus vecinas en una ventana, así que capta parecidos de uso como `doctors` y `consultants`.  
Su limitación es que cercanía significa contextos compartidos, no significado ni verdad: `good` queda a 0,94 de `bad` y `tesla` a 0,82 de `billionaire`.  
Además depende de la frecuencia: con menos de 20 apariciones, los vecinos casi no se repiten ni cambiando solo la semilla.

## Entrega

Antes de entregar:

- ejecuten todas las celdas;
- no dejen `pass`, `TODO` o errores;
- conserven resultados y gráficas;
- no cambien los nombres de las funciones solicitadas;
- entreguen el archivo `.ipynb`.
